# client

> Client for interacting with the Fewsats API

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
import os
import httpx
from typing import Dict, Any, List
import json
from fastcore.basics import BasicRepr
from fastcore.utils import store_attr
from typing import List, Dict, Any

In [ ]:
#| hide 
from dotenv import load_dotenv
from fastcore.test import *

In [ ]:
#| hide
load_dotenv()

True

The `Fewsats` class handles authentication and provides the foundation for our API interactions.

In [ ]:
#| export
class Fewsats:
    "Client for interacting with the Fewsats API"
    def __init__(self,
                 api_key: str = None, # The API key for the Fewsats account
                 base_url: str = "https://api.fewsats.com"): # The Fewsats API base URL
        self.api_key = api_key or os.environ.get("FEWSATS_API_KEY")
        if not self.api_key:
            raise ValueError("The api_key client option must be set either by passing api_key to the client or by setting the FEWSATS_API_KEY environment variable")
        self.base_url = base_url
        self._httpx_client = httpx.Client()
        self._httpx_client.headers.update({"Authorization": f"Token {self.api_key}"})

    def _request(self,
                method: str, # The HTTP method to use
                path: str, # The path to request
                timeout: int = 10, # Timeout for the request in s
                **kwargs) -> Dict[str, Any]:
        "Makes an authenticated request to Fewsats API"
        url = f"{self.base_url}/{path}"
        return  self._httpx_client.request(method, url, timeout=timeout, **kwargs)


In [ ]:
k = os.getenv("FEWSATS_API_KEY")
fs = Fewsats(api_key=k)
k = os.getenv("FEWSATS_LOCAL_API_KEY")
fs = Fewsats(api_key=k, base_url="http://localhost:8000")

test_eq(fs.api_key, k)
test_eq(fs._httpx_client.headers["Authorization"], f"Token {k}")

## Methods

### User Info

In [ ]:
#| export

@patch
def me(self: Fewsats):
    "Retrieve the user's info."
    return self._request("GET", "v0/users/me")


In [ ]:
r = fs.me()
r.status_code, r.json()

(200,
 {'name': 'Pol',
  'last_name': 'Alvarez Vecino',
  'email': 'pol@fewsats.com',
  'billing_info': None,
  'id': 1,
  'created_at': '2024-08-20T16:13:01.255Z',
  'webhook_url': 'https://example.com',
  'test_webhook_url': None})

### Balance 

In [ ]:
#| export 

@patch
def balance(self: Fewsats):
    "Retrieve the balance of the user's wallet. Amounts are always in USD cents."
    return self._request("GET", "v0/wallets")


In [ ]:
r = fs.balance()
r.status_code, r.json()

(200, [{'id': 1, 'balance': 4406, 'currency': 'usd'}])

### Payment Methods

Retrieve the user's payment methods. Useful for checking which card will be used for purchases.

In [ ]:
#| export
@patch
def payment_methods(self: Fewsats) -> List[Dict[str, Any]]:
    "Retrieve the user's payment methods, raises an exception for error status codes."
    return self._request("GET", "v0/stripe/payment-methods")


In [ ]:
r = fs.payment_methods()
payment_methods = r.json()
r.status_code, payment_methods

(200,
 [{'id': 1,
   'last4': '4242',
   'brand': 'visa',
   'exp_month': 12,
   'exp_year': 2034,
   'is_default': False},
  {'id': 4,
   'last4': '4242',
   'brand': 'Visa',
   'exp_month': 12,
   'exp_year': 2034,
   'is_default': True}])

In [ ]:
assert isinstance(payment_methods, list)

### Preview a Purchase

Preview the resulting state of a purchase. Useful, for example, to check if a CC charge is needed or the purchase will use the balance.

In [ ]:
#| export

@patch
def _preview_payment(self: Fewsats,
                    amount: str): # The amount in USD cents
    "Simulates a purchase, raises an exception for error status codes."
    assert amount.isdigit()
    return self._request("POST", "v0/l402/preview/purchase/amount", json={"amount_usd": amount})


In [ ]:
r = fs._preview_payment(amount="300") # 3.00 USD
preview = r.json()
r.status_code = preview

### Create offers

How to use the client to generate L402 offers

In [ ]:
#| export
@patch
def create_offers(self:Fewsats,
                 offers:List[Dict[str,Any]], # List of offer objects following OfferCreateV0 schema
) -> dict:
    "Create offers for L402 payment server"
    return self._request("POST", "v0/l402/offers", json={"offers": offers})

In [ ]:
test_offers = [{
    "id": "test_offer_2",
    "amount": 1,
    "currency": "usd" ,
    "description": "Test offer",
    "title": "Test Package",
    "payment_methods": ["lightning", "credit_card"]
}]

r = fs.create_offers(test_offers)
l402_offers = r.json()
r.status_code, l402_offers

(200,
 {'offers': [{'id': 'test_offer_2',
    'amount': 1,
    'currency': 'usd',
    'description': 'Test offer',
    'title': 'Test Package',
    'payment_methods': ['lightning', 'credit_card'],
    'type': 'one-off'}],
  'payment_context_token': '879e5bc7-1730-4f84-aff8-d7ab7f9bc5f4',
  'payment_request_url': 'http://localhost:8000/v0/l402/payment-request',
  'version': '0.2.2'})

### Get Payment Details

Get payment details is a convenience method for buyers to retrieve the payment information like stripe checkout url, lightning invoice etc... It does not need to be used by vendors. We demonstrate it here to showcase how an offer generated by a vendor can be turned into actual payments details.

In [ ]:
#| export
@patch
def get_payment_details(self:Fewsats,
                       payment_request_url:str, # The payment request URL
                       offer_id:str, # The offer ID
                       payment_method:str, # The payment method (lightning, credit_card, ...)
                       payment_context_token:str, # The payment context token
                       ) -> dict:
    """Gets payment details for a specific offer. Use this as buyer when you want to make the payment manually."""
    data = {"offer_id": offer_id, "payment_method": payment_method, "payment_context_token": payment_context_token}
    return httpx.post(payment_request_url, json=data)


In [ ]:
r = fs.get_payment_details(l402_offers["payment_request_url"], l402_offers["offers"][0]["id"], "lightning", l402_offers["payment_context_token"])
payment_details = r.json()
ln_invoice = payment_details["payment_request"]['lightning_invoice']
r.status_code, payment_details

(200,
 {'expires_at': '2025-03-24T15:49:44.734295+00:00',
  'offer_id': 'test_offer_2',
  'payment_request': {'lightning_invoice': 'lnbc110n1pn7zah8pp5tkszc02ez363xz2w7cv9x3m8fxw83qnxthmmyx7jfjkeq60wvslsdq523jhxapq2pskx6mpvajscqzpgxqrzpjrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5up4lek4hfwtr48uh2zkqtl7zv0xpcptfcnqtqzdz7u50vaqp6pts9qxpqysgqx6xdtfzpax6hw7ewde7u0rvys9jyypwf5ym23gcw4f2svzcpm8z3gdcr2sf4j96jazlh0cndgtql9h8ypg0c0hmfjkv5evwp0kymzmqqn8alqc'},
  'version': '0.2.2'})

### Get Payment Status

In [ ]:
#| export
@patch
def get_payment_status(self:Fewsats, 
                       payment_context_token:str, # The payment context token
                       ) -> dict:
    """Gets the status of a submitted payment. 
    Vendors should use this to check if anyone has paid for their offer associated with the token."""
    return self._request("GET", f"v0/l402/payment-status?payment_context_token={payment_context_token}")

In [ ]:

r = fs.get_payment_status(l402_offers["payment_context_token"])
r.status_code, r.json()

(400,
 {'detail': 'Invalid payment request received. You can only check the status of a test payment with a test API key'})

## Set webhook

Use this as a vendor to get notified when someone pays for your offer. Currently only 1 webhook is supported per user.

In [ ]:
#| export
@patch
def set_webhook(self:Fewsats,
                       webhook_url:str,
                       ) -> dict:
    """Set the URL where you want to receive webhooks when you receive a payment.
    Currently only 1 webhook is supported per user."""
    return self._request("POST", f"v0/users/webhook/set", json={"webhook_url": webhook_url})

In [ ]:

r = fs.set_webhook("https://example.com")
r, r.json()

(<Response [200 OK]>,
 {'name': 'Pol',
  'last_name': 'Alvarez Vecino',
  'email': 'pol@fewsats.com',
  'billing_info': None,
  'id': 1,
  'created_at': '2024-08-20T16:13:01.255Z',
  'webhook_url': 'https://example.com',
  'test_webhook_url': None})

### Pay Lightning Invoice

Pay lightning invoice is a low-level method to manually pay for a lightning invoice. 

In [ ]:
#| export

@patch
def pay_lightning(self: Fewsats, 
                  invoice: str, # lightning invoice
                  amount: int, # amount in cents
                  currency: str = "usd", # currency
                  description: str = "" ): # description of the payment 
    "Pay for a lightning invoice directly."
    data = {
        "invoice": invoice,
        "amount": amount,
        "currency": currency,
        "description": description
    }
    return self._request("POST", "v0/l402/purchases/lightning", json=data)

In [ ]:
r = fs.pay_lightning(invoice=ln_invoice,
                     description="fewsats webhook trial", amount=1)
lightning_payment = r.json()
r.status_code, lightning_payment

(200,
 {'id': '9b6a2840-b5eb-41f8-9714-023344c657df',
  'created_at': '2025-03-24T15:14:49.063Z',
  'status': 'success',
  'payment_request_url': '',
  'payment_context_token': '',
  'invoice': 'lnbc110n1pn7zah8pp5tkszc02ez363xz2w7cv9x3m8fxw83qnxthmmyx7jfjkeq60wvslsdq523jhxapq2pskx6mpvajscqzpgxqrzpjrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5up4lek4hfwtr48uh2zkqtl7zv0xpcptfcnqtqzdz7u50vaqp6pts9qxpqysgqx6xdtfzpax6hw7ewde7u0rvys9jyypwf5ym23gcw4f2svzcpm8z3gdcr2sf4j96jazlh0cndgtql9h8ypg0c0hmfjkv5evwp0kymzmqqn8alqc',
  'preimage': '3214dfbd2b499b0eae610e7aef729451e962d0bf63d07874fe08ef70e86ca39c',
  'amount': 1,
  'currency': 'usd',
  'payment_method': 'lightning',
  'title': '',
  'description': 'fewsats webhook trial',
  'type': '',
  'is_test': False})

### Pay Offer

The pay method pays for a specific offer. The user is not required to fetch the payment details beforehand. It is asynchronous and returns the `payment_id` and `status`. Using the `payment_id` we can check the status of the payment.


There are two versions of the pay offer. One accepts the L402 offers as a string, which is more convenient for text-based AI agents like LLMs. The other accepts either the `L402Offers` custom class - supported by some libraries like [Claudette](https://claudette.answer.ai/core.html#tool) - or a simple `Dict`.

In [ ]:
#| export

class Offer(BasicRepr):
    "Represents a single L402 offer"
    def __init__(self, 
                 id: str,
                 amount: int,
                 currency: str,
                 description: str,
                 title: str,
                 payment_methods: List[str] = None,
                 type: str = "one-off"): 
        store_attr()

    def __repr__(self):
        return f"Offer: {self.title}\nID: {self.id}\nAmount: {self.amount/100} {self.currency}\nDescription: {self.description}"
    
    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> 'Offer':
        "Create an Offer from a dictionary"
        return cls(**d)

class L402Offers(BasicRepr):
    "Represents the complete L402 offers schema"
    def __init__(self, 
                 offers: List[Offer],
                 payment_context_token: str,
                 payment_request_url: str,
                 version: str): 
        store_attr()
    
    def __repr__(self):
        offers_str = "\n".join([f"- {o.title} ({o.amount/100} {o.currency})" for o in self.offers])
        return f"L402 Offers:\n{offers_str}\nPayment URL: {self.payment_request_url}\nContext Token: {self.payment_context_token}"
    
    def as_dict(self) -> Dict[str, Any]:
        "Convert to dictionary format for API usage"
        return {
            'offers': [vars(o) for o in self.offers],
            'payment_context_token': self.payment_context_token,
            'payment_request_url': self.payment_request_url,
            'version': self.version
        }
    
    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> 'L402Offers':
        "Create an L402Offers object from a dictionary"
        offers = [Offer.from_dict(o) for o in d['offers']]
        return cls(
            offers=offers,
            payment_context_token=d['payment_context_token'],
            payment_request_url=d['payment_request_url'],
            version=d['version']
        )


In [ ]:
l402 = L402Offers.from_dict(l402_offers)
l402

L402 Offers:
- Test Package (0.01 usd)
Payment URL: http://localhost:8000/v0/l402/payment-request
Context Token: 879e5bc7-1730-4f84-aff8-d7ab7f9bc5f4

In [ ]:
# we make sure that invalid offers fail
invalid_json = {
    'offers': [{'id': 'test_offer_2', 'amount': 1}],  # Missing fields
    'payment_context_token': '60a8e027-8b8b-4ccf-b2b9-380ed0930283'
    # Missing payment_request_url
}
test_fail(lambda: Offer(id="test", amount=1), 
          contains="missing 3 required positional arguments: 'currency', 'description', and 'title'")


### Pay Offer


In [ ]:
#| export 

@patch
def pay_offer(self:Fewsats,
        offer_id : str, # the offer id to pay for
        l402_offer: L402Offers, # a dictionary containing L402 offers
) -> dict: # payment status response
    """Pays an offer_id from the l402_offers. 
    The l402_offer parameter must be a dictionary with this structure:
    {
        'offers': [
            {
                'id': 'test_offer_2',  # String identifier for the offer
                'amount': 1,                 # USD cents
                'currency': 'usd',           # Currency code
                'description': 'Test offer', # Text description
                'title': 'Test Package'      # Title of the package
            }
        ],
        'payment_context_token': 'token',  # Payment context token
        'payment_request_url': 'https://api.fewsats.com/v0/l402/payment-request',  # Payment URL
        'version': '0.2.2'  # API version
    }
    Returns payment status response"""
    if isinstance(l402_offer, dict): l402_offer = L402Offers.from_dict(l402_offer)
    offer_dict = l402_offer.as_dict()
    data = {"offer_id": offer_id, **offer_dict}
    return self._request("POST", "v0/l402/purchases/from-offer", json=data)


In [ ]:
offer_id = l402.offers[0].id
result = fs.pay_offer(offer_id, l402)

### Pay Offer with JSON string

This alternative method accepts a JSON string containing L402 offers. It should be used by systems that do not support custom classes in tool calling.

In [ ]:
#| export

@patch
def pay_offer_str(self:Fewsats,
        offer_id : str, # the offer id to pay for
        l402_offer: str, # JSON string containing L402 offers
) -> dict: # payment status response
    """Pays an offer_id from the l402_offers.

    The l402_offer parameter must be a JSON string with this structure:
    {
        'offers': [
            {
                'offer_id': 'test_offer_2',  # String identifier for the offer
                'amount': 1,                 # Numeric cost value
                'currency': 'usd',           # Currency code
                'description': 'Test offer', # Text description
                'title': 'Test Package'      # Title of the package
            }
        ],
        'payment_context_token': '60a8e027-8b8b-4ccf-b2b9-380ed0930283',  # Payment context token
        'payment_request_url': 'https://api.fewsats.com/v0/l402/payment-request',  # Payment URL
        'version': '0.2.2'  # API version
    }

    Returns payment status response"""
    # Parse JSON string to dictionary
    try:
        offer_data = json.loads(l402_offer)
        L402Offers.from_dict(offer_data) # we don't care about the return value, just validating the json input
    except json.JSONDecodeError:
        raise ValueError("Invalid JSON string provided for l402_offer")
    
    # Create payload with offer_id
    data = {"offer_id": offer_id, **offer_data}
    
    return self._request("POST", "v0/l402/purchases/from-offer", timeout=20, json=data)

In [ ]:
r = fs.pay_offer_str(l402_offers["offers"][0]["id"], json.dumps(l402_offers))
payment_response = r.json() if r.is_success else r.text
r.status_code, payment_response

(200,
 {'id': '99b2fc9a-06da-4ce3-b905-d2b5f3955da0',
  'created_at': '2025-03-24T15:15:01.101Z',
  'status': 'success',
  'payment_method': 'lightning'})


### Pay Link

In [ ]:
#| export

@patch
def pay_link(self:Fewsats,
        url: str, # URL to purchase from
        description: str, # Description of the purchase
        price: int, # Price in USD cents
        payment_method: str = "credit_card", # Payment method (credit_card, lightning, etc.)
        title: str = "Purchase from URL" # Title of the purchase
) -> dict: # payment status response
    """Creates a purchase record for an external URL.
    
    Args:
        url: The URL to purchase from
        description: Description of the purchase
        price: Price in USD cents
        payment_method: Payment method to use (default: credit_card)
        title: Title of the purchase (default: "Purchase from URL")
        
    Returns:
        Payment status response containing information about the purchase
    """
    data = {
        "url": url,
        "description": description,
        "price": price,
        "payment_method": payment_method,
        "title": title
    }
    return self._request("POST", "v0/l402/purchases/from-link", json=data)

In [ ]:
url = "https://www.amazon.com/Matter-Culture-Iain-M-Banks/dp/0316005371"
r = fs.pay_link(url, "Matter (Culture)", 629)
r.status_code, r.text

(200,
 '{"id": "8fdf7dfa-3b5f-463d-993c-bd18c8ca54c3", "created_at": "2025-03-25T04:01:49.668Z", "status": "pending", "payment_method": "credit_card"}')

### Payment Info

As a buyer, we can check the status of a payment as follows:

In [ ]:
#| export
@patch
def payment_info(self:Fewsats,
                  pid:str): # purchase id
    "Retrieve the details of a payment."
    return self._request("GET", f"v0/l402/outgoing-payments/{pid}")

In [ ]:
r = fs.payment_info(payment_response['id'])
r.status_code, r.json()

(200,
 {'id': 468,
  'created_at': '2025-03-19T03:40:21.890Z',
  'status': 'success',
  'payment_request_url': 'https://api.fewsats.com/v0/l402/payment-request',
  'payment_context_token': 'e0510057-481f-4abd-afb6-81374b54b897',
  'invoice': 'lnbc120n1pna5099pp5e8qmm0u36z26q7sk59zlyausychz5kt7cx3exnpln4z5qa2zay5qdq523jhxapq2pskx6mpvajscqzpgxqrzpjrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp55y4yqmcwvjy4n7kxrjtzm2lnyrpyeqljld9k28jjsrascetgyarq9qxpqysgquwacl7ppcfye6h2psk5lc8dcmgzsdrpnfgmmtg964dnt5scfpws9t4avh9rgapgy9v94maear9e94ydl8g6y80qqejt2hjfzcss0c5qpzfnjzf',
  'preimage': '91d30efdf1e1721aaf8806c0bde29fd5b5e8e6bf1738db7458ae76b720574f67',
  'amount': 1,
  'currency': 'usd',
  'payment_method': 'lightning',
  'title': 'Test Package',
  'description': 'Test offer',
  'type': 'one-off',
  'is_test': False})

## As tools

In [ ]:
#| export

@patch
def as_tools(self:Fewsats):
    "Return list of available tools for AI agents"
    return [
        self.me,
        self.balance,
        self.payment_methods,
        self.pay_offer_str,
        self.payment_info,
    ]

In [ ]:
fs.as_tools()

[<bound method Fewsats.me of <__main__.Fewsats object>>,
 <bound method Fewsats.balance of <__main__.Fewsats object>>,
 <bound method Fewsats.payment_methods of <__main__.Fewsats object>>,
 <bound method Fewsats.pay_offer_str of <__main__.Fewsats object>>,
 <bound method Fewsats.payment_info of <__main__.Fewsats object>>]

Both the preview and purchase methods automatically use the default payment method if a charge is needed. This client provides a straightforward way to interact with the Fewsats API, making it easy for developers to integrate Fewsats functionality into their applications.

## Agent Demo

We will use [Claudette](https://claudette.answer.ai/) to demonstrate how to pay for content using the Fewsats API.

In [ ]:
from claudette import Chat, models

In [ ]:
model = models[1]
model

'claude-3-5-sonnet-20240620'

In [ ]:
fs.balance()

<Response [200 OK]>

In [ ]:
chat = Chat(model, sp='You are a helpful assistant that can pay offers.', tools=fs.as_tools())
pr = f"Could you pay the cheapest offer in {l402_offers}?"
r = chat.toolloop(pr, trace_func=print)
r

Message(id='msg_01EpyhaUvHwEipp6FULhBYie', content=[TextBlock(text="Certainly! I'll analyze the offer information you provided and proceed with paying the cheapest offer. In this case, there's only one offer available, so we'll pay for that one.\n\nFirst, let's check your balance to ensure you have sufficient funds, and then we'll proceed with the payment.", type='text'), ToolUseBlock(id='toolu_01GWCtvpcbbZczXqyfnEKwRu', input={}, name='balance', type='tool_use')], model='claude-3-5-sonnet-20240620', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=In: 1038; Out: 98; Cache create: 0; Cache read: 0; Total: 1136)
Message(id='msg_01PJTa29HyuUNfHs6oWkm5QW', content=[TextBlock(text="I apologize, but it seems the balance information wasn't returned in a format I can interpret. Let's proceed with the payment, and if there are any issues with insufficient funds, the system will let us know.\n\nNow, let's pay for the offer:", type='text'), ToolUseBlock(id='too

I apologize again, but it seems I'm unable to interpret the balance information from the response. 

Given the difficulties we're encountering, here are a few suggestions:

1. Check your account balance manually to ensure you have sufficient funds for the purchase.
2. Verify that the offer details are correct and still valid.
3. Try the payment again later, as there might be a temporary system issue.

If you'd like to try the payment again or if you have any other questions or concerns, please let me know, and I'll be happy to assist you further.

<details>

- id: `msg_01BLwjeuhoNB669DEgCwsCHW`
- content: `[{'text': "I apologize again, but it seems I'm unable to interpret the balance information from the response. \n\nGiven the difficulties we're encountering, here are a few suggestions:\n\n1. Check your account balance manually to ensure you have sufficient funds for the purchase.\n2. Verify that the offer details are correct and still valid.\n3. Try the payment again later, as there might be a temporary system issue.\n\nIf you'd like to try the payment again or if you have any other questions or concerns, please let me know, and I'll be happy to assist you further.", 'type': 'text'}]`
- model: `claude-3-5-sonnet-20240620`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 1570, 'output_tokens': 126}`

</details>

We can see in the chat history to see that the agent correctlye filled the required information for the payment.

The payment balance has also decreased as expected.

In [ ]:
fs.balance(), chat.h

(<Response [200 OK]>,
 [{'role': 'user',
   'content': [{'type': 'text',
     'text': "Could you pay the cheapest offer in {'offers': [{'id': 'test_offer_2', 'amount': 1, 'currency': 'usd', 'description': 'Test offer', 'title': 'Test Package', 'payment_methods': ['lightning', 'credit_card'], 'type': 'one-off'}], 'payment_context_token': 'e0510057-481f-4abd-afb6-81374b54b897', 'payment_request_url': 'https://api.fewsats.com/v0/l402/payment-request', 'version': '0.2.2'}?"}]},
  {'role': 'assistant',
   'content': [TextBlock(text="Certainly! I'll analyze the offer information you provided and proceed with paying the cheapest offer. In this case, there's only one offer available, so we'll pay for that one.\n\nFirst, let's check your balance to ensure you have sufficient funds, and then we'll proceed with the payment.", type='text'),
    ToolUseBlock(id='toolu_01GWCtvpcbbZczXqyfnEKwRu', input={}, name='balance', type='tool_use')]},
  {'role': 'user',
   'content': [{'type': 'tool_result',
 

In [ ]:
#|hide
from nbdev.doclinks import nbdev_export
nbdev_export()